# Phase 6 | Decision Report

**Project**: Beyond Risk Scores: Uplift-Driven Financial Intervention for Loan Default Prevention

This report consolidates findings from five phases of analysis into a single recommendation.
Every number in this report was derived from data. Every claim is backed by statistical
evidence. The goal is one clear decision: deploy, target, extend, or stop.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

COLORS = ['steelblue', 'cadetblue', 'skyblue', 'lightblue']

## Step 1 | Load Final Data

In [2]:
df = pd.read_csv('../data/processed/segmented_scores.csv')

In [3]:
persuadables = df[df['quadrant'] == 'Persuadables'].copy()
sure_things = df[df['quadrant'] == 'Sure Things'].copy()
sleeping_dogs = df[df['quadrant'] == 'Sleeping Dogs'].copy()

print("=" * 45)
print("Final Data Loaded")
print("=" * 45)
print(f"Total customers        : {len(df):,}")
print(f"Persuadables           : {len(persuadables):,}")
print(f"Sure Things            : {len(sure_things):,}")
print(f"Sleeping Dogs          : {len(sleeping_dogs):,}")
print()
print(f"Treatment default rate : {df[df['TREATMENT']==1]['TARGET'].mean()*100:.2f}%")
print(f"Control default rate   : {df[df['TREATMENT']==0]['TARGET'].mean()*100:.2f}%")
print(f"Observed effect        : {(df[df['TREATMENT']==0]['TARGET'].mean() - df[df['TREATMENT']==1]['TARGET'].mean())*100:.2f} pp")
print("=" * 45)

Final Data Loaded
Total customers        : 71,904
Persuadables           : 29,131
Sure Things            : 26,865
Sleeping Dogs          : 15,908

Treatment default rate : 6.92%
Control default rate   : 9.78%
Observed effect        : 2.86 pp


## Step 2 | Executive Summary

This section is written for a senior stakeholder with five minutes and a decision to make.
No technical jargon. No model names. Just the problem, the finding, and the money.

### The Problem

The collections team calls every at-risk customer when they show signs of financial stress.
This costs money and agent time. But not every customer responds the same way to a call.
Some recover on their own without help. Some default no matter what. And some actually get
worse when contacted. The only customers worth calling are those who will recover specifically
because of the call.

### What We Found

We analyzed 105,366 at-risk customers and identified three distinct groups:

- **40.5% are Persuadables (29,131 customers)**: These customers will likely default without
  a call but recover when contacted. Calling them reduces their default rate from 20.75% to
  6.84%. Every call to this group is worth an average of 44,000 CU in recovered capital.

- **37.4% are Sure Things (26,865 customers)**: These customers recover on their own. Their
  default rate is below 3% with or without a call. Calling them wastes agent time.

- **22.1% are Sleeping Dogs (15,908 customers)**: These customers get worse when contacted.
  Their default rate jumps from 2.92% to 16.19% after a call. Every call to this group
  destroys value.

### The Bottom Line

Calling only Persuadables instead of calling everyone delivers:
- 60% fewer calls (29,131 vs 71,904)
- 60% lower cost (1.5M CU vs 3.6M CU)
- 2.7x more net value (937M CU vs 352M CU)

The current approach of calling all at-risk customers is leaving hundreds of millions in
value on the table and actively harming 22% of the customers it contacts.

## Step 3 | Evidence Review

Six phases of analysis produced this recommendation. This section summarizes the key
evidence from each phase in plain language.

**Phase 1 | Exploratory Data Analysis**
Analyzed 307,511 customers. Portfolio default rate is 8.07%. Identified 105,366 at-risk
customers (DTI >= 0.20) with a higher default rate of 8.58%. Primary risk drivers are
external credit scores, age, and debt-to-income ratio.

**Phase 2 | Experiment Design**
Needed 2,738 customers per group to detect a 2 pp effect. Had 52,683 available (19.2x
surplus). Matched 35,952 treatment-control pairs using Propensity Score Matching. All
covariates balanced below SMD 0.01. Post-experiment power confirmed at 1.00.

**Phase 3 | A/B Test**
Intervention reduced defaults by 2.86 pp. Statistically significant (p < 0.001) with
95% confidence interval of [2.46 pp, 3.26 pp]. The effect clears both the financial
break-even threshold (0.0145%) and the operational MDE (2.0 pp) by wide margins.

**Phase 4 | Uplift Modeling**
Built three models (T-Learner, X-Learner, Causal Forest). T-Learner ranked first.
Identified 29,131 Persuadables (40.5%), 26,865 Sure Things (37.4%), and 15,908 Sleeping
Dogs (22.1%). Targeting only Persuadables delivers 2.7x more value than calling everyone.

**Phase 5 | Segmentation**
Every segment (income, age, employment, education, loan type) is profitable. No exclusions
needed. Priority matrix ranks Persuadables by uplift and loan value. Top cell (High Uplift,
High Value) returns 111,340 CU per call. Bottom cell still returns 2,053 CU per call.

## Step 4 | ROI Calculator

This calculator allows a collections manager to input their team capacity and cost
assumptions and see exactly how many customers to target, in what order, and what the
expected return will be.

Change the three inputs below and re-run the cell to see updated results.

In [6]:
# ROI CALCULATOR - CHANGE THESE THREE INPUTS
total_agents = 10          # Number of collection agents
calls_per_agent_per_day = 25  # Calls each agent can make per day
cost_per_call = 50 

In [7]:
# Calculations
daily_capacity = total_agents * calls_per_agent_per_day
lgd = 0.45

In [8]:
# Rank Persuadables by uplift (highest first)
ranked = persuadables.sort_values('uplift_t_learner', ascending=False).reset_index(drop=True)
ranked['recovery_value'] = ranked['AMT_CREDIT'] * lgd
ranked['expected_gain'] = (ranked['uplift_t_learner'] * ranked['recovery_value']) - cost_per_call
ranked['cumulative_gain'] = ranked['expected_gain'].cumsum()
ranked['day'] = (ranked.index // daily_capacity) + 1

total_days = int(np.ceil(len(ranked) / daily_capacity))
total_cost = len(ranked) * cost_per_call
peak_gain = ranked['cumulative_gain'].max()
optimal_count = ranked['cumulative_gain'].idxmax() + 1

In [9]:
print("=" * 60)
print("ROI CALCULATOR")
print("=" * 60)
print()
print("INPUTS")
print(f"  Collection agents      : {total_agents}")
print(f"  Calls per agent/day    : {calls_per_agent_per_day}")
print(f"  Daily capacity         : {daily_capacity} calls")
print(f"  Cost per call          : {cost_per_call} CU")
print()
print("TARGETING RECOMMENDATION")
print(f"  Customers to call      : {optimal_count:,} Persuadables")
print(f"  Customers to skip      : {len(df) - optimal_count:,} (Sure Things + Sleeping Dogs)")
print(f"  Days to complete       : {total_days}")
print()
print("EXPECTED RETURNS")
print(f"  Total intervention cost: {total_cost:,.0f} CU")
print(f"  Expected net gain      : {peak_gain:,.0f} CU")
print(f"  ROI                    : {(peak_gain / total_cost) * 100:,.0f}%")
print()
print("DAILY BREAKDOWN (first 5 days)")
print(f"{'Day':<6} {'Calls':>8} {'Cumulative Gain':>20}")
print("-" * 36)
for day in range(1, min(6, total_days + 1)):
    day_end = min(day * daily_capacity, len(ranked)) - 1
    print(f"{day:<6} {daily_capacity:>8} {ranked['cumulative_gain'].iloc[day_end]:>20,.0f} CU")
print()

# What percentage of portfolio to target
pct_targeted = optimal_count / len(df) * 100
print(f"BOTTOM LINE: Call the top {pct_targeted:.1f}% of the portfolio")
print(f"to maximize ROI. Calling more leads to diminishing returns.")
print("=" * 60)

ROI CALCULATOR

INPUTS
  Collection agents      : 10
  Calls per agent/day    : 25
  Daily capacity         : 250 calls
  Cost per call          : 50 CU

TARGETING RECOMMENDATION
  Customers to call      : 29,106 Persuadables
  Customers to skip      : 42,798 (Sure Things + Sleeping Dogs)
  Days to complete       : 117

EXPECTED RETURNS
  Total intervention cost: 1,456,550 CU
  Expected net gain      : 683,509,550 CU
  ROI                    : 46,927%

DAILY BREAKDOWN (first 5 days)
Day       Calls      Cumulative Gain
------------------------------------
1           250           48,260,806 CU
2           250           81,896,986 CU
3           250          111,695,765 CU
4           250          137,786,696 CU
5           250          161,090,804 CU

BOTTOM LINE: Call the top 40.5% of the portfolio
to maximize ROI. Calling more leads to diminishing returns.


### ROI Calculator Finding

With a team of 10 agents making 25 calls per day, the collections team can reach all 29,106
Persuadables in 117 working days. The first 5 days alone generate 161M CU in cumulative
gains because the highest-uplift customers are called first.

The bottom line: call the top 40.5% of the at-risk portfolio. Skip the remaining 59.5%.
The 42,798 customers being skipped are either Sure Things (who recover without help) or
Sleeping Dogs (who get worse when called). Every call to those groups is either wasted
or harmful.

The manager can adjust the three inputs (agents, calls per day, cost per call) and re-run
to model different team sizes and budget scenarios.

## Step 5 | Decision Framework

The business problem document defined four possible outcomes. Each has a clear condition.
The data determines which one applies.

| Decision           | Condition                                                    |
|--------------------|--------------------------------------------------------------|
| Full Rollout       | Positive ROI across all tested segments                      |
| Targeted Rollout   | Profitable only for specific segments                        |
| Extend Experiment  | Statistical power too low to make a confident call           |
| Do Not Deploy      | Intervention cost exceeds expected recovery value            |

### Decision Framework Evaluation

**Do Not Deploy**: Does not apply. The observed uplift of 2.86 pp is 197x above the
break-even threshold of 0.0145%. The intervention is overwhelmingly profitable.

**Extend Experiment**: Does not apply. Achieved power is 1.00. The p-value is below 0.001.
The confidence interval [2.46 pp, 3.26 pp] does not include zero. The result is statistically
robust beyond doubt.

**Targeted Rollout**: Does not apply. Every segment tested in Phase 5 is profitable. All 4
income bands, all 5 age groups, all 4 employment types, and all 4 education levels deliver
positive net value per call. No segment needs to be excluded.

**Full Rollout**: Applies. All conditions are met. Positive ROI across all segments,
sufficient statistical power, significant effect, and effect far above break-even.

### Decision: Full Rollout

Target all 29,131 Persuadables across all segments. Skip all 26,865 Sure Things and avoid
all 15,908 Sleeping Dogs. Use the priority matrix from Phase 5 to determine call order.

## Step 6 | Final Recommendation

### What to do

Deploy the uplift-driven collections strategy targeting all 29,131 Persuadables. Use the
priority matrix to determine call order: High Uplift, High Value customers first (1,330
customers at 111,340 CU per call), then work down the ranking until agent capacity runs out.

### What not to do

Do not call the 26,865 Sure Things. They recover on their own. Calling them costs 1.3M CU
in agent time with negligible impact on default rates.

Do not call the 15,908 Sleeping Dogs. Contacting them increases their default rate from
2.92% to 16.19%. Every call to this group destroys value and damages the customer
relationship.

### Financial justification

| Metric                  | Uplift Strategy     | Current Strategy (Call All) |
|-------------------------|--------------------|-----------------------------|
| Customers called        | 29,131             | 71,904                      |
| Intervention cost       | 1,456,550 CU       | 3,595,200 CU                |
| Net value               | 683,509,550 CU     | 352,096,403 CU              |
| ROI                     | 46,927%            | 9,793%                      |

The uplift strategy calls 60% fewer customers, costs 60% less, and delivers nearly 2x the
net value. The improvement comes from two sources: concentrating calls on customers who
respond (Persuadables) and eliminating calls that cause harm (Sleeping Dogs).

### What this means for the collections team

A 10-agent team making 25 calls per day completes the full Persuadable list in 117 working
days. The highest-value customers are called in the first two weeks. By day 5, cumulative
gains exceed 161M CU. There is no scenario in the data where calling Sure Things or
Sleeping Dogs improves the outcome.

## Step 7 | Implementation Roadmap

### Phase A | Weeks 1-2: Containerize and Deploy

- Package the uplift scoring pipeline into a Python module under src/
- Containerize using Docker for environment reproducibility
- Set up a FastAPI endpoint that accepts customer features and returns uplift score,
  quadrant assignment, and priority rank
- Write unit tests for the scoring logic
- Deploy to a staging environment for internal testing

### Phase B | Month 1: Canary Rollout (5%)

- Select the top 1,456 Persuadables (5% of 29,131) from the priority matrix
- Run the intervention on this group while holding a matched control set
- Monitor for Sleeping Dog behavior: if any sub-segment shows increased defaults after
  contact, pause and investigate before scaling
- Compare observed uplift against the 2.86 pp benchmark from Phase 3
- Success criteria: positive uplift with no evidence of harm in any segment

### Phase C | Month 2: Validation Rollout (20%)

- Expand to the top 5,826 Persuadables
- Confirm statistical significance of the observed effect at this scale
- Validate that the priority matrix ranking holds: higher-ranked customers should show
  stronger uplift than lower-ranked ones
- Begin building the Plotly Dash monitoring dashboard

### Phase D | Month 3+: Full Scale and Monitoring

- Scale to all 29,131 Persuadables
- Deploy the Plotly Dash dashboard to track uplift score distributions, segment-level
  treatment effects, and recovery rates across rollout stages
- Monitor for model drift: if customer behavior shifts due to economic changes, the
  uplift scores need recalibration
- Retrain the model quarterly using new outcome data to keep scores current
- Establish alerts for segments where uplift drops below break-even or where Sleeping
  Dog behavior emerges in previously safe segments

## Step 8 | Project Conclusion

This project started with a simple question: among customers showing financial stress, who
will recover only because we reached out to them?

Six phases of analysis produced a clear answer. 40.5% of at-risk customers are Persuadables
who respond to intervention. 37.4% are Sure Things who recover on their own. 22.1% are
Sleeping Dogs who get worse when contacted.

A traditional risk-score model cannot make this distinction. It ranks customers by how likely
they are to default. It does not rank them by how likely they are to respond. The result is a
collections strategy that wastes 60% of its calls and actively harms 22% of the customers it
contacts.

The uplift-driven strategy recommended in this report calls fewer customers, costs less, and
recovers more. Every number in this report was derived from the data. Every recommendation
is backed by statistical evidence and financial justification.

The recommendation is full rollout across all Persuadable segments, deployed through a staged
canary process that validates the model at 5%, 20%, and 100% before committing full resources.